In [ ]:

# **Cell 1**
from google.colab import drive
import os, re, json, math, random, subprocess, shutil, glob, sys
from pathlib import Path
from PIL import Image

drive.mount('/content/drive', force_remount=False)
FOLDER='/content/drive/MyDrive/AIgenerated'
AUDIO_PATH=os.path.join(FOLDER,'trimmed_audio.m4a')
SONG_MAP_PATH=os.path.join(FOLDER,'song_map.json')
ASSETS_ROOT=os.path.join(FOLDER,'assets')
FONTS_ROOT=os.path.join(ASSETS_ROOT,'fonts')
EMOJI_ROOT=os.path.join(ASSETS_ROOT,'emoji')
OUTPUT_ROOT=os.path.join(FOLDER,'stage2_outputs')
WORK_ROOT=os.path.join(FOLDER,'_stage2_work')
os.makedirs(OUTPUT_ROOT,exist_ok=True); os.makedirs(WORK_ROOT,exist_ok=True)
print('✅ Drive mounted; Stage 2 workspace ready.')

Mounted at /content/drive
✅ Drive mounted; Stage 2 workspace ready.


In [ ]:

# **Cell 2**
if not os.path.isfile(AUDIO_PATH): raise FileNotFoundError(AUDIO_PATH)
if not os.path.isfile(SONG_MAP_PATH): raise FileNotFoundError(SONG_MAP_PATH)
with open(SONG_MAP_PATH,'r',encoding='utf-8') as f: SONG_MAP=json.load(f)
LINES=SONG_MAP.get('lines',[])
if not LINES: raise RuntimeError('No lines in song_map.json')
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',AUDIO_PATH],capture_output=True,text=True,check=True)
AUDIO_DURATION=float(probe.stdout.strip())
print('STAGE 1 VALIDATION'); print('==================')
print('Lines:',len(LINES)); print(f'Audio duration: {AUDIO_DURATION:.3f}s')
print('Source timing map loaded; no alignment/Whisper work will run.')

STAGE 1 VALIDATION
Lines: 9
Audio duration: 36.560s
Source timing map loaded; no alignment/Whisper work will run.


In [ ]:
# **Cell 3**
def natural_key(p):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)',Path(p).name)]
image_paths=sorted([os.path.join(FOLDER,x) for x in os.listdir(FOLDER) if Path(x).suffix.lower() in {'.png','.jpg','.jpeg','.webp'} and not x.startswith('_')],key=natural_key)
VALID_IMAGES=[]
for p in image_paths:
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im: VALID_IMAGES.append({'path':p,'name':Path(p).name,'size':im.size})
    except Exception as e: print('⚠️ skipped',Path(p).name,e)
if not VALID_IMAGES: raise RuntimeError('No valid images in AIgenerated/.')
print('Images available:',len(VALID_IMAGES))
for i,x in enumerate(VALID_IMAGES,1): print(f'{i:02d}. {x["name"]} {x["size"][0]}x{x["size"][1]}')

Images available: 9
01. 1SM_Cozy Bedroom Daydream in Pastels.jpg 2103x1683
02. 2SM_Reaching Across a Cozy Bedroom.jpg 2103x1683
03. 3SM_Peaceful Bedroom Meditation Moment.jpg 2103x1683
04. 4SM_Dreamy Bedroom Memories.jpg 2103x1683
05. 5SM_IMG_20260902_122351.jpg 1362x1570
06. 6SM_Cozy Nighttime Bedroom Reflection.jpg 2172x1629
07. 7SM_Rainy Night Bedroom Retreat.jpg 2172x1629
08. 8SM_Cozy Bedroom Pout and Vanity.jpg 2172x1629
09. 9SM_Stuck in a Cozy Nighttime Thought.jpg 2172x1629


In [ ]:
# ============================================================
# CELL 4 — UNIFIED VISUAL GROUP PARSER
# ============================================================
# Replace original Cell 4.
# A physical lyric line = one visual group/image slot.
# '|' = separate lyric section inside that same visual group.

PIPE_GROUPS=[]
MAPPED=[]

USE_COUNT=min(len(VALID_IMAGES),len(LINES))
if len(VALID_IMAGES)<len(LINES):
    print(f'⚠️ {len(LINES)-len(VALID_IMAGES)} physical lyric line(s) have no image and will be discarded.')
    for z in LINES[len(VALID_IMAGES):]: print('  discarded:',z.get('line',z))
elif len(VALID_IMAGES)>len(LINES):
    print(f'ℹ️ {len(VALID_IMAGES)-len(LINES)} extra image(s) ignored.')

for i in range(USE_COUNT):
    ln=LINES[i]
    im=VALID_IMAGES[i]
    raw_line=str(ln.get('line',''))
    sections=[s.strip() for s in raw_line.split('|') if s.strip()]

    group={
        'visual_group':i+1,
        'image_slot':i+1,
        'slot':i+1,
        'image_path':im['path'],
        'image_name':im['name'],
        'line_no':ln.get('line_no',i+1),
        'sections_text':sections,
    }
    PIPE_GROUPS.append(group)
    MAPPED.append({
        'slot':i+1,
        'line':ln,
        'image_path':im['path'],
        'image_name':im['name'],
        'visual_group':i+1,
    })

print('UNIFIED VISUAL GROUP CHECK')
print('===========================')
for g in PIPE_GROUPS:
    print(f"Group {g['visual_group']} → image slot {g['image_slot']} | {g['image_name']}")
    for j,s in enumerate(g['sections_text'],1):
        print(f'  {j:02d}. {s}')

UNIFIED VISUAL GROUP CHECK
Group 1 → image slot 1 | 1SM_Cozy Bedroom Daydream in Pastels.jpg
  01. Memories follow me left and right
Group 2 → image slot 2 | 2SM_Reaching Across a Cozy Bedroom.jpg
  01. I can feel you over here, I can feel you over here
Group 3 → image slot 3 | 3SM_Peaceful Bedroom Meditation Moment.jpg
  01. You take up every corner of my mind
  02. (What you gon' do now?)
Group 4 → image slot 4 | 4SM_Dreamy Bedroom Memories.jpg
  01. Ever since the d-day y-you went away
  02. (No, I don't know how)
Group 5 → image slot 5 | 5SM_IMG_20260902_122351.jpg
  01. How to erase your body from out my brain
  02. (What you gon' do now?)
Group 6 → image slot 6 | 6SM_Cozy Nighttime Bedroom Reflection.jpg
  01. Maybe I should just focus on me instead
  02. (But all I think about)
Group 7 → image slot 7 | 7SM_Rainy Night Bedroom Retreat.jpg
  01. All the nights we were tangled up in your bed
Group 8 → image slot 8 | 8SM_Cozy Bedroom Pout and Vanity.jpg
  01. Oh, no (oh, no)
  02. O

In [ ]:
# ============================================================
# CELL 5 — UNIFIED SECTION TIMELINE
# ============================================================
# Replace original Cell 5.
# Section boundaries come from the existing Stage-1 timed words.
# The | separator itself has no timing.

MAP_START=float(LINES[0]['start'])
TIMELINE=[]

for group in PIPE_GROUPS:
    source_li=next((x for x in LINES if x.get('line_no')==group['line_no']),None)
    if source_li is None:
        continue

    timed_words=[w for w in source_li.get('words',[]) if str(w.get('word','')).strip()]
    sections=[]
    cursor=0

    for section_no,section_text in enumerate(group['sections_text'],1):
        # Emoji are not Stage-1 words, so count only non-emoji lexical pieces.
        # Use a local helper if Cell 13 has not run yet.
        cleaned=re.sub(r'[\U0001F000-\U0001FAFF\u2600-\u27BF\uFE0F\u200D\U0001F3FB-\U0001F3FF]', '', section_text)
        n=len(cleaned.split())
        if n<=0:
            continue

        selected=timed_words[cursor:cursor+n]
        if not selected:
            continue
        a=cursor
        cursor+=len(selected)

        start=float(selected[0]['start'])
        end=float(selected[-1]['end'])
        sections.append({
            'section_no':section_no,
            'text':section_text,
            'start':start,
            'end':end,
            'duration':max(0.0,end-start),
            'word_range':[a,cursor],
        })

    TIMELINE.append({
        **group,
        'sections':sections,
    })

print('UNIFIED TIMELINE CHECK')
print('=======================')
for group in TIMELINE:
    print(f"Image {group['image_slot']} | {group['image_name']} | {len(group['sections'])} section(s)")
    for sec in group['sections']:
        print(f"  {sec['section_no']:02d}. {sec['start']-MAP_START:.3f} → {sec['end']-MAP_START:.3f} | {sec['text']}")

UNIFIED TIMELINE CHECK
Image 1 | 1SM_Cozy Bedroom Daydream in Pastels.jpg | 1 section(s)
  01. 0.000 → 2.640 | Memories follow me left and right
Image 2 | 2SM_Reaching Across a Cozy Bedroom.jpg | 1 section(s)
  01. 2.640 → 5.400 | I can feel you over here, I can feel you over here
Image 3 | 3SM_Peaceful Bedroom Meditation Moment.jpg | 2 section(s)
  01. 5.400 → 7.560 | You take up every corner of my mind
  02. 7.560 → 8.700 | (What you gon' do now?)
Image 4 | 4SM_Dreamy Bedroom Memories.jpg | 2 section(s)
  01. 8.700 → 11.020 | Ever since the d-day y-you went away
  02. 11.020 → 12.520 | (No, I don't know how)
Image 5 | 5SM_IMG_20260902_122351.jpg | 2 section(s)
  01. 13.540 → 17.060 | How to erase your body from out my brain
  02. 17.060 → 18.360 | (What you gon' do now?)
Image 6 | 6SM_Cozy Nighttime Bedroom Reflection.jpg | 2 section(s)
  01. 18.360 → 21.640 | Maybe I should just focus on me instead
  02. 21.630 → 21.640 | (But all I think about)
Image 7 | 7SM_Rainy Night Bedroom Ret

In [ ]:
# ============================================================
# CELL 6 — ONE ANIMATION PER VISUAL GROUP
# ============================================================
# Replace original Cell 6.
RNG=random.SystemRandom()
ANIMATIONS=['subtle_zoom_in','subtle_zoom_out','pan_left','pan_right','pan_up','pan_down','micro_drift']
for g in TIMELINE:
    g['animation']=RNG.choice(ANIMATIONS)
print('Animation plan: one animation per visual group/image.')
for g in TIMELINE:
    print(g['image_name'],g['animation'])

Animation plan: one animation per visual group/image.
1SM_Cozy Bedroom Daydream in Pastels.jpg subtle_zoom_out
2SM_Reaching Across a Cozy Bedroom.jpg pan_right
3SM_Peaceful Bedroom Meditation Moment.jpg pan_right
4SM_Dreamy Bedroom Memories.jpg pan_right
5SM_IMG_20260902_122351.jpg subtle_zoom_in
6SM_Cozy Nighttime Bedroom Reflection.jpg subtle_zoom_out
7SM_Rainy Night Bedroom Retreat.jpg subtle_zoom_in
8SM_Cozy Bedroom Pout and Vanity.jpg subtle_zoom_in
9SM_Stuck in a Cozy Nighttime Thought.jpg subtle_zoom_in


In [ ]:

# **Cell 7**
OUTPUT_W,OUTPUT_H=1080,1920
FPS=30
print(f'Output: {OUTPUT_W}x{OUTPUT_H} @ {FPS}fps')
print('Policy: crop to 9:16; never non-uniformly stretch images.')

Output: 1080x1920 @ 30fps
Policy: crop to 9:16; never non-uniformly stretch images.


In [ ]:

# **Cell 8**
LYRIC_MODE='LINE'
print('Lyric mode:',LYRIC_MODE)

Lyric mode: LINE


In [ ]:
# ============================================================
# CELL 9 — UNIFIED RENDER PLAN
# ============================================================
# Replace original Cell 9.
PLAN_PATH=os.path.join(WORK_ROOT,'stage2_render_plan.json')
plan={
    'version':4,
    'audio':AUDIO_PATH,
    'song_map':SONG_MAP_PATH,
    'width':OUTPUT_W,
    'height':OUTPUT_H,
    'fps':FPS,
    'lyric_mode':LYRIC_MODE,
    'timeline':TIMELINE,
}
with open(PLAN_PATH,'w',encoding='utf-8') as f:
    json.dump(plan,f,ensure_ascii=False,indent=2)
print('✅ Unified render plan saved:',PLAN_PATH)

✅ Unified render plan saved: /content/drive/MyDrive/AIgenerated/_stage2_work/stage2_render_plan.json


In [ ]:
# ============================================================
# CELL 10 — UNIFIED INPUT CHECK
# ============================================================
# Replace original Cell 10.
print('UNIFIED STAGE 2 CHECK')
print('=====================')
print('Physical lyric lines:',len(LINES))
print('Visual groups:',len(TIMELINE))
print('Images:',len(VALID_IMAGES))
print('Audio:',f'{AUDIO_DURATION:.3f}s')
print('Mode:',LYRIC_MODE)
for g in TIMELINE:
    print(f"  Image {g['image_slot']}: {len(g['sections'])} section(s)")
print('✅ Unified Stage 2 inputs ready.')

UNIFIED STAGE 2 CHECK
Physical lyric lines: 9
Visual groups: 9
Images: 9
Audio: 36.560s
Mode: LINE
  Image 1: 1 section(s)
  Image 2: 1 section(s)
  Image 3: 2 section(s)
  Image 4: 2 section(s)
  Image 5: 2 section(s)
  Image 6: 2 section(s)
  Image 7: 1 section(s)
  Image 8: 2 section(s)
  Image 9: 2 section(s)
✅ Unified Stage 2 inputs ready.


In [ ]:
# ============================================================
# CELL 10B — UNIFIED RESUME MODE
# ============================================================
# Replace original Cell 10B.
# Rebuilds unified grouping/timeline state without Stage-1 alignment.

from google.colab import drive
import os,re,json,math,random,subprocess,glob
from pathlib import Path
from PIL import Image

drive.mount('/content/drive',force_remount=False)
FOLDER='/content/drive/MyDrive/AIgenerated'
AUDIO_PATH=os.path.join(FOLDER,'trimmed_audio.m4a')
SONG_MAP_PATH=os.path.join(FOLDER,'song_map.json')
ASSETS_ROOT=os.path.join(FOLDER,'assets')
FONTS_ROOT=os.path.join(ASSETS_ROOT,'fonts')
EMOJI_ROOT=os.path.join(ASSETS_ROOT,'emoji')
OUTPUT_ROOT=os.path.join(FOLDER,'stage2_outputs')
WORK_ROOT=os.path.join(FOLDER,'_stage2_work')
BASE_VIDEO=os.path.join(OUTPUT_ROOT,'stage2_base.mp4')
os.makedirs(OUTPUT_ROOT,exist_ok=True); os.makedirs(WORK_ROOT,exist_ok=True)

if not os.path.isfile(AUDIO_PATH): raise FileNotFoundError(AUDIO_PATH)
if not os.path.isfile(SONG_MAP_PATH): raise FileNotFoundError(SONG_MAP_PATH)
with open(SONG_MAP_PATH,'r',encoding='utf-8') as f: SONG_MAP=json.load(f)
LINES=SONG_MAP.get('lines',[])
if not LINES: raise RuntimeError('song_map.json contains no lyric lines.')
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',AUDIO_PATH],capture_output=True,text=True,check=True)
AUDIO_DURATION=float(probe.stdout.strip())

def natural_key(p):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)',Path(p).name)]
image_paths=sorted([os.path.join(FOLDER,x) for x in os.listdir(FOLDER) if Path(x).suffix.lower() in {'.png','.jpg','.jpeg','.webp'} and not x.startswith('_')],key=natural_key)
VALID_IMAGES=[]
for p in image_paths:
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im: VALID_IMAGES.append({'path':p,'name':Path(p).name,'size':im.size})
    except Exception as e: print('⚠️ skipped',Path(p).name,e)
if not VALID_IMAGES: raise RuntimeError('No valid images in AIgenerated/.')

PIPE_GROUPS=[]
MAPPED=[]
for i in range(min(len(VALID_IMAGES),len(LINES))):
    ln=LINES[i]; im=VALID_IMAGES[i]
    sections=[s.strip() for s in str(ln.get('line','')).split('|') if s.strip()]
    PIPE_GROUPS.append({'visual_group':i+1,'image_slot':i+1,'slot':i+1,'image_path':im['path'],'image_name':im['name'],'line_no':ln.get('line_no',i+1),'sections_text':sections})
    MAPPED.append({'slot':i+1,'line':ln,'image_path':im['path'],'image_name':im['name'],'visual_group':i+1})

MAP_START=float(LINES[0]['start'])
TIMELINE=[]
for group in PIPE_GROUPS:
    li=next((x for x in LINES if x.get('line_no')==group['line_no']),None)
    if not li: continue
    tw=[w for w in li.get('words',[]) if str(w.get('word','')).strip()]
    cursor=0; sections=[]
    for sno,text in enumerate(group['sections_text'],1):
        cleaned=re.sub(r'[\U0001F000-\U0001FAFF\u2600-\u27BF\uFE0F\u200D\U0001F3FB-\U0001F3FF]','',text)
        n=len(cleaned.split())
        selected=tw[cursor:cursor+n]; a=cursor; cursor+=len(selected)
        if not selected: continue
        sections.append({'section_no':sno,'text':text,'start':float(selected[0]['start']),'end':float(selected[-1]['end']),'duration':max(0,float(selected[-1]['end'])-float(selected[0]['start'])),'word_range':[a,cursor]})
    TIMELINE.append({**group,'sections':sections})

RNG=random.SystemRandom(); ANIMATIONS=['subtle_zoom_in','subtle_zoom_out','pan_left','pan_right','pan_up','pan_down','micro_drift']
for g in TIMELINE: g['animation']=RNG.choice(ANIMATIONS)
OUTPUT_W,OUTPUT_H=1080,1920; FPS=30
LYRIC_MODE=globals().get('LYRIC_MODE','WORD')
print('🔄 UNIFIED RESUME MODE'); print('======================'); print('Physical lines:',len(LINES)); print('Visual groups:',len(TIMELINE)); print('Images:',len(VALID_IMAGES)); print(f'Audio duration: {AUDIO_DURATION:.3f}s'); print('Lyric mode:',LYRIC_MODE); print('✅ Unified Stage 2 state restored. Continue with Cell 11B or Cell 11.')

Mounted at /content/drive
🔄 UNIFIED RESUME MODE
Physical lines: 9
Visual groups: 9
Images: 9
Audio duration: 36.560s
Lyric mode: WORD
✅ Unified Stage 2 state restored. Continue with Cell 11B or Cell 11.


In [ ]:
# ============================================================
# CELL 11 — UNIFIED BASE VIDEO / FULL-DURATION VISUAL GROUPS
# ============================================================
# Replace original Cell 11.
# For a group, its image stays active from group start until the next
# visual-group start, including gaps and all | sections. Final group holds
# through the full audio duration.

BASE_VIDEO=os.path.join(OUTPUT_ROOT,'stage2_base.mp4')

def _pipe_prep(im,t,dur,anim):
    im=im.convert('RGB'); sw,sh=im.size; tr=OUTPUT_W/OUTPUT_H; sr=sw/sh
    if sr>tr:
        cw=int(sh*tr); l=(sw-cw)//2; im=im.crop((l,0,l+cw,sh))
    else:
        ch=int(sw/tr); top=(sh-ch)//2; im=im.crop((0,top,sw,top+ch))
    im=im.resize((OUTPUT_W,OUTPUT_H),Image.Resampling.LANCZOS)
    p=max(0,min(1,t/max(dur,.001))); scale=1
    if anim=='subtle_zoom_in': scale=1+.035*p
    elif anim=='subtle_zoom_out': scale=1.035-.035*p
    elif anim in ('pan_left','pan_right','pan_up','pan_down','micro_drift'): scale=1.025
    if scale!=1:
        nw,nh=int(OUTPUT_W*scale),int(OUTPUT_H*scale); im=im.resize((nw,nh),Image.Resampling.LANCZOS)
        if anim=='pan_left': x=int((nw-OUTPUT_W)*p)
        elif anim=='pan_right': x=int((nw-OUTPUT_W)*(1-p))
        else: x=(nw-OUTPUT_W)//2
        if anim=='pan_up': y=int((nh-OUTPUT_H)*p)
        elif anim=='pan_down': y=int((nh-OUTPUT_H)*(1-p))
        else: y=(nh-OUTPUT_H)//2
        im=im.crop((x,y,x+OUTPUT_W,y+OUTPUT_H))
    return im

def _group_start_seconds(group):
    secs=group.get('sections',[])
    if not secs: return None
    return min(float(s['start']) for s in secs)-float(LINES[0]['start'])

group_spans=[]
for gi,g in enumerate(TIMELINE):
    gs=_group_start_seconds(g)
    if gs is None: continue
    if gi+1<len(TIMELINE):
        ns=_group_start_seconds(TIMELINE[gi+1])
        ge=ns if ns is not None else float(AUDIO_DURATION)
    else:
        ge=float(AUDIO_DURATION)
    group_spans.append((g,max(0,gs),max(0,min(float(AUDIO_DURATION),ge))))

if not group_spans: raise RuntimeError('No timed visual groups found.')
first_group,first_start,_=group_spans[0]
if first_start>0: group_spans.insert(0,(first_group,0.0,first_start))

# Important: if a single visual group owns many pipe sections, it remains the
# same image continuously across all gaps. Group span ends only at next group.
from PIL import Image
import tempfile
segment_paths=[]
concat_file=os.path.join(WORK_ROOT,'unified_base_concat.txt')
for idx,(g,gs,ge) in enumerate(group_spans):
    dur=max(.001,ge-gs)
    pth=os.path.join(WORK_ROOT,f'unified_group_{idx:03d}.jpg')
    with Image.open(g['image_path']) as im:
        _pipe_prep(im,0,dur,g.get('animation','micro_drift')).save(pth,quality=95)
    segment_paths.append((pth,dur))

with open(concat_file,'w',encoding='utf-8') as f:
    for pth,dur in segment_paths:
        f.write(f"file '{pth}'\n")
        f.write(f'duration {dur:.6f}\n')
    f.write(f"file '{segment_paths[-1][0]}'\n")

subprocess.run(['ffmpeg','-y','-v','error','-f','concat','-safe','0','-i',concat_file,'-vf',f'fps={FPS},format=yuv420p','-t',str(AUDIO_DURATION),'-c:v','libx264','-preset','veryfast','-crf','20',BASE_VIDEO],check=True)
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',BASE_VIDEO],capture_output=True,text=True,check=True)
print('✅ Unified base video saved:',BASE_VIDEO)
print('Base duration:',float(probe.stdout.strip()))
print('Expected audio duration:',AUDIO_DURATION)
print('Visual groups:',len(group_spans))

✅ Unified base video saved: /content/drive/MyDrive/AIgenerated/stage2_outputs/stage2_base.mp4
Base duration: 36.567
Expected audio duration: 36.56
Visual groups: 9


In [ ]:
# ============================================================
# CELL 11B — UNIFIED RESUME BASE CHECKPOINT
# ============================================================
# Replace original Cell 11B.
# Existing base video may be reused, but unified RAM state is rebuilt.

from google.colab import drive
import os,re,json,subprocess,glob
from pathlib import Path
from PIL import Image

drive.mount('/content/drive',force_remount=False)
FOLDER='/content/drive/MyDrive/AIgenerated'
AUDIO_PATH=os.path.join(FOLDER,'trimmed_audio.m4a')
SONG_MAP_PATH=os.path.join(FOLDER,'song_map.json')
ASSETS_ROOT=os.path.join(FOLDER,'assets'); FONTS_ROOT=os.path.join(ASSETS_ROOT,'fonts'); EMOJI_ROOT=os.path.join(ASSETS_ROOT,'emoji')
OUTPUT_ROOT=os.path.join(FOLDER,'stage2_outputs'); WORK_ROOT=os.path.join(FOLDER,'_stage2_work'); BASE_VIDEO=os.path.join(OUTPUT_ROOT,'stage2_base.mp4')
os.makedirs(OUTPUT_ROOT,exist_ok=True); os.makedirs(WORK_ROOT,exist_ok=True)

if not os.path.isfile(AUDIO_PATH): raise FileNotFoundError(AUDIO_PATH)
if not os.path.isfile(SONG_MAP_PATH): raise FileNotFoundError(SONG_MAP_PATH)
if not os.path.isfile(BASE_VIDEO): raise FileNotFoundError('❌ stage2_base.mp4 not found. Run unified Cell 11 first.')
with open(SONG_MAP_PATH,'r',encoding='utf-8') as f: SONG_MAP=json.load(f)
LINES=SONG_MAP.get('lines',[])
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',AUDIO_PATH],capture_output=True,text=True,check=True)
AUDIO_DURATION=float(probe.stdout.strip())

def natural_key(p): return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)',Path(p).name)]
image_paths=sorted([os.path.join(FOLDER,x) for x in os.listdir(FOLDER) if Path(x).suffix.lower() in {'.png','.jpg','.jpeg','.webp'} and not x.startswith('_')],key=natural_key)
VALID_IMAGES=[]
for p in image_paths:
    try:
        with Image.open(p) as im: im.verify()
        with Image.open(p) as im: VALID_IMAGES.append({'path':p,'name':Path(p).name,'size':im.size})
    except Exception as e: print('⚠️ skipped',Path(p).name,e)

PIPE_GROUPS=[]; MAPPED=[]
for i in range(min(len(VALID_IMAGES),len(LINES))):
    ln=LINES[i]; im=VALID_IMAGES[i]; sections=[s.strip() for s in str(ln.get('line','')).split('|') if s.strip()]
    PIPE_GROUPS.append({'visual_group':i+1,'image_slot':i+1,'slot':i+1,'image_path':im['path'],'image_name':im['name'],'line_no':ln.get('line_no',i+1),'sections_text':sections})
    MAPPED.append({'slot':i+1,'line':ln,'image_path':im['path'],'image_name':im['name'],'visual_group':i+1})
MAP_START=float(LINES[0]['start']); TIMELINE=[]
for g in PIPE_GROUPS:
    li=next((x for x in LINES if x.get('line_no')==g['line_no']),None)
    if not li: continue
    tw=[w for w in li.get('words',[]) if str(w.get('word','')).strip()]; cursor=0; sections=[]
    for sno,text in enumerate(g['sections_text'],1):
        cleaned=re.sub(r'[\U0001F000-\U0001FAFF\u2600-\u27BF\uFE0F\u200D\U0001F3FB-\U0001F3FF]','',text); n=len(cleaned.split()); a=cursor; selected=tw[cursor:cursor+n]; cursor+=len(selected)
        if not selected: continue
        sections.append({'section_no':sno,'text':text,'start':float(selected[0]['start']),'end':float(selected[-1]['end']),'duration':max(0,float(selected[-1]['end'])-float(selected[0]['start'])),'word_range':[a,cursor]})
    TIMELINE.append({**g,'sections':sections})
RNG=random.SystemRandom(); ANIMATIONS=['subtle_zoom_in','subtle_zoom_out','pan_left','pan_right','pan_up','pan_down','micro_drift']
for g in TIMELINE: g['animation']=RNG.choice(ANIMATIONS)
OUTPUT_W,OUTPUT_H=1080,1920; FPS=30; LYRIC_MODE=globals().get('LYRIC_MODE','WORD')
base_probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration:stream=width,height','-of','json',BASE_VIDEO],capture_output=True,text=True,check=True)
BASE_META=json.loads(base_probe.stdout)
print('🔄 UNIFIED CELL 11B RESUME'); print('=========================='); print('Physical lines:',len(LINES)); print('Visual groups:',len(TIMELINE)); print('Images:',len(VALID_IMAGES)); print(f'Audio duration: {AUDIO_DURATION:.3f}s'); print('Base video:',BASE_VIDEO); print(json.dumps(BASE_META,indent=2)); print('Lyric mode:',LYRIC_MODE); print('✅ Resume state ready. Continue with Cell 12 → Cell 13.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 UNIFIED CELL 11B RESUME
Physical lines: 1
Visual groups: 1
Images: 1
Audio duration: 9.060s
Base video: /content/drive/MyDrive/AIgenerated/stage2_outputs/stage2_base.mp4
{
  "programs": [],
  "streams": [
    {
      "width": 1080,
      "height": 1920
    }
  ],
  "format": {
    "duration": "9.067000"
  }
}
Lyric mode: WORD
✅ Resume state ready. Continue with Cell 12 → Cell 13.


In [ ]:

# **Cell 12**
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration:stream=width,height','-of','json',BASE_VIDEO],capture_output=True,text=True,check=True)
print('BASE VIDEO CHECK'); print(json.dumps(json.loads(probe.stdout),indent=2)); print('✅ Base checkpoint valid.')

BASE VIDEO CHECK
{
  "programs": [],
  "streams": [
    {
      "width": 1080,
      "height": 1920
    }
  ],
  "format": {
    "duration": "36.567000"
  }
}
✅ Base checkpoint valid.


In [ ]:

# CELL 13 — UNIFIED PIPE + EMOJI + LINE/WORD RENDERER v2
# ============================================================
# Built directly against the user's working Stage 2 structure.
# Keeps Stage 1, Cell 11B/base video, Cell 12, Cell 15, Cell 16 untouched.
#
# This renderer fixes the previous PIPE renderer problems:
# • no regex emoji parser
# • explicit RGB bytes before FFmpeg writer
# • writer stderr is collected and surfaced if FFmpeg exits
# • LINE mode remains one timed event per pipe section
# • WORD mode uses one render state per frame (no duplicate alpha layers)
# • WORD placement is deterministic and local to the SAME image area
# • LINE wrapping is frame-safe and includes emoji width
# • exact emoji filename matching; no fuzzy prefix matches
#
# Prerequisite variables come from unified Cells 4/5/6/10B/11B.
# Run this as a full replacement for original Cell 13 only.

from PIL import Image, ImageDraw, ImageFont
import os, glob, json, math, random, subprocess
from pathlib import Path

# ------------------------------------------------------------
# REQUIRED STATE
# ------------------------------------------------------------
required=['OUTPUT_ROOT','WORK_ROOT','FONTS_ROOT','EMOJI_ROOT','AUDIO_PATH',
          'AUDIO_DURATION','BASE_VIDEO','OUTPUT_W','OUTPUT_H','FPS',
          'LYRIC_MODE','LINES','TIMELINE']
missing=[k for k in required if k not in globals()]
if missing:
    raise RuntimeError('Missing Stage 2 variables: '+', '.join(missing))

LYRIC_VIDEO=os.path.join(OUTPUT_ROOT,'stage2_lyrics.mp4')
os.makedirs(WORK_ROOT,exist_ok=True)

# ------------------------------------------------------------
# FONT DISCOVERY
# ------------------------------------------------------------
FONT_FILES=[]
for root,ds,fs in os.walk(FONTS_ROOT):
    FONT_FILES += [os.path.join(root,f) for f in fs if f.lower().endswith(('.ttf','.otf'))]
if not FONT_FILES:
    FONT_FILES=glob.glob('/usr/share/fonts/**/*.ttf',recursive=True)
if not FONT_FILES:
    raise RuntimeError('No font files found.')

def choose_font(size):
    pool=FONT_FILES[:20].copy()
    random.shuffle(pool)
    for f in pool:
        try:
            return ImageFont.truetype(f,size)
        except Exception:
            pass
    return ImageFont.truetype(FONT_FILES[0],size)

# ------------------------------------------------------------
# EMOJI DISCOVERY + EXACT LOOKUP
# ------------------------------------------------------------
EMOJI_FILES=[]
for root,ds,fs in os.walk(EMOJI_ROOT):
    EMOJI_FILES += [os.path.join(root,f) for f in fs if f.lower().endswith(('.png','.webp'))]
EMOJI_INDEX={Path(p).stem.lower():p for p in EMOJI_FILES}

print(f'Fonts available: {len(FONT_FILES)}')
print(f'Emoji assets available: {len(EMOJI_FILES)}')
print('Lyric mode:',LYRIC_MODE)

def emoji_codepoints(ch):
    return '-'.join(f'{ord(c):x}' for c in ch)

def emoji_path(ch):
    if not ch:
        return None
    cp=emoji_codepoints(ch)
    p=EMOJI_INDEX.get(cp)
    if p:
        return p
    # compatibility fallback only when the asset differs solely by FE0F
    clean=''.join(c for c in ch if ord(c)!=0xfe0f)
    if clean!=ch:
        return EMOJI_INDEX.get(emoji_codepoints(clean))
    return None

# ------------------------------------------------------------
# EMOJI SCANNER — NO REGEX
# ------------------------------------------------------------
BASE_RANGES=[(0x1F1E6,0x1F1FF),(0x1F300,0x1FAFF),(0x2600,0x27BF)]
SKIN=(0x1F3FB,0x1F3FF)
VS16=0xFE0F
ZWJ=0x200D

def is_emoji_base(ch):
    cp=ord(ch)
    return any(a<=cp<=b for a,b in BASE_RANGES)

def consume_emoji(text,i):
    if i>=len(text) or not is_emoji_base(text[i]):
        return None
    j=i+1
    if j<len(text) and ord(text[j])==VS16:
        j+=1
    if j<len(text) and SKIN[0]<=ord(text[j])<=SKIN[1]:
        j+=1
    while j<len(text) and ord(text[j])==ZWJ:
        j+=1
        if j>=len(text) or not is_emoji_base(text[j]):
            break
        j+=1
        if j<len(text) and ord(text[j])==VS16:
            j+=1
        if j<len(text) and SKIN[0]<=ord(text[j])<=SKIN[1]:
            j+=1
    return text[i:j],j

def extract_emoji(text):
    out=[]
    i=0
    while i<len(text):
        got=consume_emoji(text,i)
        if got:
            ch,j=got
            out.append((i,j,ch))
            i=j
        else:
            i+=1
    # adjacent regional indicators = one flag token
    merged=[]
    for item in out:
        if merged and len(merged[-1][2])==1 and len(item[2])==1:
            a,b=merged[-1][2],item[2]
            if 0x1F1E6<=ord(a)<=0x1F1FF and 0x1F1E6<=ord(b)<=0x1F1FF:
                merged[-1]=(merged[-1][0],item[1],a+b)
                continue
        merged.append(item)
    return merged

def remove_emoji(text):
    matches=extract_emoji(text)
    if not matches:
        return text
    out=[]
    last=0
    for s,e,ch in matches:
        out.append(text[last:s])
        last=e
    out.append(text[last:])
    return ''.join(out)

def attach_emojis_to_words(line):
    text=str(line or '')
    matches=extract_emoji(text)
    cleaned=text
    for s,e,ch in reversed(matches):
        cleaned=cleaned[:s]+' '+cleaned[e:]
    words=cleaned.split()
    attachments=[None]*len(words)
    for s,e,ch in matches:
        prev=remove_emoji(text[:s]).split()
        if not prev:
            continue
        target=prev[-1]
        for i in range(len(words)-1,-1,-1):
            if words[i]==target:
                attachments[i]=(attachments[i] or '')+ch
                break
    return words,attachments

# ------------------------------------------------------------
# BUILD EVENTS FROM UNIFIED TIMELINE
# ------------------------------------------------------------
MAP_START=float(LINES[0]['start'])
word_events=[]

if str(LYRIC_MODE).upper()=='LINE':
    for group in TIMELINE:
        group_image=group.get('image_slot',group.get('slot',1))
        for sec in group.get('sections',[]):
            text=str(sec.get('text','')).strip()
            if not text:
                continue
            plain,attachments=attach_emojis_to_words(text)
            word_events.append({
                'start':max(0.0,float(sec['start'])-MAP_START),
                'end':max(0.0,float(sec['end'])-MAP_START),
                'text':' '.join(plain),
                'emoji_tokens':[(i,e) for i,e in enumerate(attachments) if e],
                'image_slot':group_image,
                'visual_group':group.get('visual_group',group_image),
                'section_no':sec.get('section_no',1)
            })
else:
    for group in TIMELINE:
        source_li=next((x for x in LINES if x.get('line_no')==group.get('line_no')),None)
        if source_li is None:
            continue
        group_image=group.get('image_slot',group.get('slot',1))
        timed_words=[w for w in source_li.get('words',[]) if str(w.get('word','')).strip()]
        for sec in group.get('sections',[]):
            a,b=sec['word_range']
            section_words=timed_words[a:b]
            plain,attachments=attach_emojis_to_words(sec.get('text',''))
            for pos,w in enumerate(section_words):
                raw=str(w.get('word','')).strip()
                if not raw:
                    continue
                ws=float(w['start'])-MAP_START
                we=float(w['end'])-MAP_START
                word_events.append({
                    'start':max(0.0,ws),
                    'end':max(ws,we),
                    'text':raw,
                    'emoji':attachments[pos] if pos<len(attachments) else None,
                    'image_slot':group_image,
                    'visual_group':group.get('visual_group',group_image),
                    'section_no':sec.get('section_no',1)
                })

print('\nUNIFIED EMOJI/PIPE EVENT CHECK')
print('===============================')
for ev in word_events:
    if str(LYRIC_MODE).upper()=='LINE':
        print(f"{ev['start']:.3f}->{ev['end']:.3f} | {ev['text']} | emoji={ev.get('emoji_tokens',[])} | image={ev.get('image_slot')} | section={ev.get('section_no')}")
    else:
        print(f"{ev['start']:.3f}->{ev['end']:.3f} | {ev['text']} | emoji={ev.get('emoji') or ''} | image={ev.get('image_slot')} | section={ev.get('section_no')}")

# ------------------------------------------------------------
# STYLES — deterministic per EVENT
# ------------------------------------------------------------
styles={}
PALETTE=[(255,255,255),(255,80,180),(0,235,255),(255,225,70),(170,100,255),(90,255,130),(255,145,60)]
for idx,ev in enumerate(word_events):
    styles[idx]={
        'size':random.choice([76,86,96,108,120]),
        'fill':random.choice(PALETTE),
        'stroke':random.choice([(0,0,0),(255,255,255),(40,20,60)]),
        'sw':random.choice([2,3,4]),
        'x':random.randint(330,750),
        'y':random.choice([500,750,1000,1250,1500]),
        'font':choose_font(random.choice([76,86,96,108,120]))
    }
for idx,ev in enumerate(word_events):
    ev['_style_index']=idx

# ------------------------------------------------------------
# LAYOUT
# ------------------------------------------------------------
MAX_LINE_CHARS=20
EMOJI_GAP=24
LINE_GAP=18

def load_emoji(emoji,font):
    ep=emoji_path(emoji)
    if not ep:
        return None
    try:
        img=Image.open(ep).convert('RGBA')
        target=max(42,int(font.size*.78))
        img.thumbnail((target,target),Image.Resampling.LANCZOS)
        return img
    except Exception:
        return None

def draw_word_unit(layer,d,word,emoji,font,fill,stroke,sw,x,y):
    bbox=d.textbbox((0,0),word,font=font,stroke_width=sw)
    ww=bbox[2]-bbox[0]
    d.text((x+3,y+3),word,font=font,fill=(0,0,0,130),stroke_width=sw+1,stroke_fill=(0,0,0,100))
    d.text((x,y),word,font=font,fill=fill,stroke_width=sw,stroke_fill=stroke)
    cursor=x+ww
    img=load_emoji(emoji,font) if emoji else None
    if img:
        cursor+=EMOJI_GAP
        layer.alpha_composite(img,(int(cursor),int(y+max(0,(font.size-img.height)//2))))
        cursor+=img.width
    return cursor

def wrap_tokens(tokens,max_chars=MAX_LINE_CHARS):
    rows=[]
    cur=[]
    count=0
    for tok in tokens:
        i,w,e=tok
        add=len(w)+(1 if cur else 0)
        if cur and count+add>max_chars:
            rows.append(cur)
            cur=[]
            count=0
            add=len(w)
        cur.append(tok)
        count+=add
    if cur:
        rows.append(cur)
    return rows

def draw_line_event(frame,ev,progress):
    st=styles[ev['_style_index']]
    font=st['font']
    layer=Image.new('RGBA',frame.size,(0,0,0,0))
    d=ImageDraw.Draw(layer)
    words=ev['text'].split()
    emap={i:e for i,e in ev.get('emoji_tokens',[])}

    # Whole line stays ONE timed event; early reveal is purely visual.
    visible_count=len(words)
    if progress<0.12 and words:
        visible_count=max(1,min(len(words),int(math.ceil(len(words)*progress/.12))))
    visible=words[:visible_count]
    tokens=[(i,w,emap.get(i)) for i,w in enumerate(visible)]
    rows=wrap_tokens(tokens)

    # A visual-safe vertical placement. If the line needs extra rows,
    # center the block so it stays inside the 1080x1920 canvas.
    block_h=max(font.size,len(rows)*font.size+max(0,len(rows)-1)*LINE_GAP)
    preferred_y=int(st['y'])
    top_y=max(30,min(OUTPUT_H-block_h-30,preferred_y))
    base_x=int(st['x'])

    for row_no,row in enumerate(rows):
        # Measure complete row including each emoji's dedicated width.
        row_width=0
        for local_i,(_,w,e) in enumerate(row):
            bb=d.textbbox((0,0),w,font=font,stroke_width=st['sw'])
            row_width+=bb[2]-bb[0]
            if e:
                img=load_emoji(e,font)
                if img:
                    row_width+=EMOJI_GAP+img.width
                    img.close()
            if local_i<len(row)-1:
                row_width+=max(12,int(font.size*.25))

        x=max(25,min(OUTPUT_W-row_width-25,base_x-row_width//2))
        y=top_y+row_no*(font.size+LINE_GAP)
        for local_i,(_,w,e) in enumerate(row):
            x=draw_word_unit(layer,d,w,e,font,st['fill'],st['stroke'],st['sw'],x,y)
            if local_i<len(row)-1:
                x+=max(12,int(font.size*.25))

    return Image.alpha_composite(frame.convert('RGBA'),layer)

# ------------------------------------------------------------
# WORD MODE — stable local position, no per-frame random movement
# ------------------------------------------------------------
WORD_POSITIONS={}

def get_word_position(ev):
    group=ev.get('visual_group',ev.get('image_slot',1))
    section=ev.get('section_no',1)
    key=(group,section)
    if key not in WORD_POSITIONS:
        # Stable area based on first event of the section.
        base=styles[ev['_style_index']]
        WORD_POSITIONS[key]={
            'x':int(base['x']),
            'y':int(base['y']),
        }
    anchor=WORD_POSITIONS[key]

    # Position determined from the event's original word order, not frame time.
    li=next((x for x in LINES if x.get('line_no')==ev.get('_source_line_no')),None)
    order=ev.get('_word_index',0)
    offsets=[(-110,-55),(110,-55),(-110,55),(110,55)]
    ox,oy=offsets[order%4]
    return anchor['x']+ox,anchor['y']+oy

# Stamp source indices once so WORD placement is deterministic.
for wi,ev in enumerate(word_events):
    if str(LYRIC_MODE).upper()=='WORD':
        # Events are generated in lyric order; calculate a per-section counter.
        same=[e for e in word_events[:wi] if e.get('visual_group')==ev.get('visual_group') and e.get('section_no')==ev.get('section_no')]
        ev['_word_index']=len(same)
        if TIMELINE:
            # Find source line through visual group/section.
            for li in LINES:
                if any(g.get('line_no')==li.get('line_no') and g.get('visual_group')==ev.get('visual_group') for g in TIMELINE):
                    ev['_source_line_no']=li.get('line_no')
                    break

def draw_word_event(frame,ev,progress):
    st=styles[ev['_style_index']]
    font=st['font']
    text=ev['text']
    emoji=ev.get('emoji') or ''
    reveal=max(1,min(len(text),int(math.ceil(len(text)*min(1,progress/.12))))) if text else 0
    shown=text[:reveal]

    layer=Image.new('RGBA',frame.size,(0,0,0,0))
    d=ImageDraw.Draw(layer)

    emoji_img=load_emoji(emoji,font) if emoji else None
    tw=d.textbbox((0,0),shown,font=font,stroke_width=st['sw'])[2]
    total_w=tw+(EMOJI_GAP+emoji_img.width if emoji_img else 0)

    x,y=get_word_position(ev)
    x=max(20,min(OUTPUT_W-total_w-20,int(x-total_w/2)))
    y=max(40,min(OUTPUT_H-font.size-40,int(y)))

    d.text((x+3,y+3),shown,font=font,fill=(0,0,0,130),stroke_width=st['sw']+1,stroke_fill=(0,0,0,100))
    d.text((x,y),shown,font=font,fill=st['fill'],stroke_width=st['sw'],stroke_fill=st['stroke'])

    if emoji_img:
        ex=x+tw+EMOJI_GAP
        ey=y+max(0,(font.size-emoji_img.height)//2)
        layer.alpha_composite(emoji_img,(int(ex),int(ey)))
        emoji_img.close()

    return Image.alpha_composite(frame.convert('RGBA'),layer)

def draw_event(frame,ev,progress):
    if str(LYRIC_MODE).upper()=='LINE':
        return draw_line_event(frame,ev,progress)
    return draw_word_event(frame,ev,progress)

# ------------------------------------------------------------
# FFMPEG — EXPLICIT RGB24 PIPE + REAL ERROR REPORTING
# ------------------------------------------------------------
reader=subprocess.Popen(
    ['ffmpeg','-v','error','-i',BASE_VIDEO,'-f','rawvideo','-pix_fmt','rgb24','-r',str(FPS),'pipe:1'],
    stdout=subprocess.PIPE,stderr=subprocess.PIPE
)

writer=subprocess.Popen(
    ['ffmpeg','-y','-v','error',
     '-f','rawvideo','-pix_fmt','rgb24',
     '-s',f'{OUTPUT_W}x{OUTPUT_H}','-r',str(FPS),
     '-i','pipe:0',
     '-i',AUDIO_PATH,
     '-map','0:v:0','-map','1:a:0',
     '-c:v','libx264','-preset','veryfast','-crf','20',
     '-pix_fmt','yuv420p',
     '-c:a','aac','-b:a','192k',
     '-t',str(AUDIO_DURATION),
     LYRIC_VIDEO],
    stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE
)

FB=OUTPUT_W*OUTPUT_H*3
idx=0
write_error=None

try:
    while True:
        raw=reader.stdout.read(FB)
        if len(raw)<FB:
            break
        t=idx/FPS
        frame=Image.frombytes('RGB',(OUTPUT_W,OUTPUT_H),raw)
        active=[e for e in word_events if e['start']<=t<e['end']]
        for ev in active:
            frame=draw_event(frame,ev,(t-ev['start'])/max(0.001,ev['end']-ev['start']))

        # Absolutely explicit: RGB mode, exactly FB bytes per frame.
        out_bytes=frame.convert('RGB').tobytes()
        if len(out_bytes)!=FB:
            raise RuntimeError(f'Frame byte mismatch: expected {FB}, got {len(out_bytes)}')
        writer.stdin.write(out_bytes)
        idx+=1

    reader.stdout.close()
    reader.wait()
    writer.stdin.close()
    fferr=writer.stderr.read().decode(errors='ignore')
    rc=writer.wait()
    if rc:
        raise RuntimeError('FFmpeg lyric render failed:\n'+fferr[-5000:])

except BrokenPipeError:
    try:
        writer.stdin.close()
    except Exception:
        pass
    fferr=writer.stderr.read().decode(errors='ignore')
    try:
        writer.wait()
    except Exception:
        pass
    raise RuntimeError('FFmpeg writer closed the pipe. Actual FFmpeg error:\n'+fferr[-5000:])
finally:
    try: reader.kill()
    except Exception: pass
    try: writer.stdin.close()
    except Exception: pass

print('✅ Unified lyric video saved:',LYRIC_VIDEO)
print('Mode:',LYRIC_MODE)
print('Events rendered:',len(word_events))
print('Audio duration:',f'{AUDIO_DURATION:.3f}s')
print('Line wrap limit:',MAX_LINE_CHARS)
print('Frame format sent to FFmpeg: RGB24')

Fonts available: 89
Emoji assets available: 3689
Lyric mode: LINE

UNIFIED EMOJI/PIPE EVENT CHECK
0.000->2.640 | Memories follow me left and right | emoji=[] | image=1 | section=1
2.640->5.400 | I can feel you over here, I can feel you over here | emoji=[] | image=2 | section=1
5.400->7.560 | You take up every corner of my mind | emoji=[] | image=3 | section=1
7.560->8.700 | (What you gon' do now?) | emoji=[] | image=3 | section=2
8.700->11.020 | Ever since the d-day y-you went away | emoji=[] | image=4 | section=1
11.020->12.520 | (No, I don't know how) | emoji=[] | image=4 | section=2
13.540->17.060 | How to erase your body from out my brain | emoji=[] | image=5 | section=1
17.060->18.360 | (What you gon' do now?) | emoji=[] | image=5 | section=2
18.360->21.640 | Maybe I should just focus on me instead | emoji=[] | image=6 | section=1
21.630->21.640 | (But all I think about) | emoji=[] | image=6 | section=2
23.080->26.580 | All the nights we were tangled up in your bed | emoji=[] | i

In [ ]:
# FINAL VIDEO CHECK
# ============================================================
probe=subprocess.run(
    ['ffprobe','-v','error','-show_entries','format=duration:stream=width,height','-of','json',LYRIC_VIDEO],
    capture_output=True,text=True,check=True
)
print('FINAL VIDEO CHECK')
print('=================')
print(json.dumps(json.loads(probe.stdout),indent=2))
print('Mode:',LYRIC_MODE)
print('Final video:',LYRIC_VIDEO)
print('Audio source:',AUDIO_PATH)
print('Song map:',SONG_MAP_PATH)
print('✅ Final render exists.')

FINAL VIDEO CHECK
{
  "programs": [],
  "streams": [
    {
      "width": 1080,
      "height": 1920
    },
    {}
  ],
  "format": {
    "duration": "36.567000"
  }
}
Mode: WORD
Final video: /content/drive/MyDrive/AIgenerated/stage2_outputs/stage2_lyrics.mp4
Audio source: /content/drive/MyDrive/AIgenerated/trimmed_audio.m4a
Song map: /content/drive/MyDrive/AIgenerated/song_map.json
✅ Final render exists.


In [ ]:

# **Cell 15**

import os
import shutil
from pathlib import Path

# ============================================================
# USER SETTINGS — change only these if needed
# ============================================================
PROJECT_NAME = 'Chal Wahn jate hain 4lines'
KEEP_SONG_MP3 = False          # DEFAULT: NO
KEEP_BASE_VIDEO = False        # DEFAULT: NO

# ============================================================
# PATHS
# ============================================================
AI_ROOT = '/content/drive/MyDrive/AIgenerated'
ARCHIVE_ROOT = '/content/drive/MyDrive/Completed video sync'

os.makedirs(ARCHIVE_ROOT, exist_ok=True)

# ============================================================
# IMPORTANT PROJECT FILES
# ============================================================
source_files = [
    ('lyrics.txt', os.path.join(AI_ROOT, 'lyrics.txt'), True),
    ('trimmed_audio.m4a', os.path.join(AI_ROOT, 'trimmed_audio.m4a'), True),
    ('song_map.json', os.path.join(AI_ROOT, 'song_map.json'), True),
    ('stage2_lyrics.mp4', os.path.join(AI_ROOT, 'stage2_outputs', 'stage2_lyrics.mp4'), True),
    ('song.mp3', os.path.join(AI_ROOT, 'song.mp3'), KEEP_SONG_MP3),
    ('stage2_base.mp4', os.path.join(AI_ROOT, 'stage2_outputs', 'stage2_base.mp4'), KEEP_BASE_VIDEO),
]

# ============================================================
# FIND A FREE ARCHIVE FOLDER — NEVER OVERWRITE
# ============================================================
base_name = str(PROJECT_NAME).strip() or 'Project'
archive_dir = os.path.join(ARCHIVE_ROOT, base_name)

counter = 2
while os.path.exists(archive_dir):
    archive_dir = os.path.join(ARCHIVE_ROOT, f'{base_name}{counter}')
    counter += 1

os.makedirs(archive_dir)

# ============================================================
# COPY + VERIFY
# ============================================================
archived = []
missing = []

for filename, src, enabled in source_files:
    if not enabled:
        continue

    if not os.path.isfile(src):
        missing.append(filename)
        continue

    dst = os.path.join(archive_dir, filename)
    shutil.copy2(src, dst)

    # Basic size verification before source cleanup.
    if os.path.getsize(src) != os.path.getsize(dst):
        raise RuntimeError(f'❌ Verification failed for {filename}. Source was NOT deleted.')

    archived.append(filename)

# Required files must exist before we touch workspace files.
required_names = ['lyrics.txt', 'trimmed_audio.m4a', 'song_map.json', 'stage2_lyrics.mp4']
missing_required = [x for x in required_names if x not in archived]

if missing_required:
    # Remove incomplete archive rather than leaving a misleading project folder.
    shutil.rmtree(archive_dir, ignore_errors=True)
    raise RuntimeError(
        '❌ Archive aborted. Required file(s) missing: '
        + ', '.join(missing_required)
    )

# ============================================================
# REMOVE COPIED PROJECT FILES FROM ACTIVE WORKSPACE
# Keep assets/ untouched.
# ============================================================
for filename in archived:
    for _, src, enabled in source_files:
        if enabled and os.path.basename(src) == filename and os.path.isfile(src):
            os.remove(src)
            break

print('================================')
print('PROJECT ARCHIVED')
print('================================')
print('Archive folder:', archive_dir)
print('Saved files:')
for x in archived:
    print('  ✅', x)

print()
print('Not archived by default:')
if not KEEP_SONG_MP3:
    print('  ❌ song.mp3')
if not KEEP_BASE_VIDEO:
    print('  ❌ stage2_base.mp4')

print()
print('⚠️ Note: this cell archives only the important project files.')
print('Run Cell 16 separately if you want a complete AIgenerated workspace reset.')

PROJECT ARCHIVED
Archive folder: /content/drive/MyDrive/Completed video sync/Chal Wahn jate hain 4lines2
Saved files:
  ✅ lyrics.txt
  ✅ trimmed_audio.m4a
  ✅ song_map.json
  ✅ stage2_lyrics.mp4

Not archived by default:
  ❌ song.mp3
  ❌ stage2_base.mp4

⚠️ Note: this cell archives only the important project files.
Run Cell 16 separately if you want a complete AIgenerated workspace reset.


In [ ]:

# CELL 16 — COMPLETE CLEANUP (DIRECT, FIXED)
# ============================================================
# Purpose:
# • Keep assets/ untouched.
# • Keep stage2_outputs/ folder itself.
# • EMPTY stage2_outputs/ contents.
# • Delete EVERYTHING ELSE inside AIgenerated/.
# • No confirmation prompt / special command.
#
# Run Cell 15 FIRST when you want to archive the project.
# Then run this Cell 16.
# ============================================================

from google.colab import drive
import os
import shutil

drive.mount('/content/drive', force_remount=False)

FOLDER='/content/drive/MyDrive/AIgenerated'
ASSETS_ROOT=os.path.join(FOLDER,'assets')
OUTPUT_ROOT=os.path.join(FOLDER,'stage2_outputs')

if not os.path.isdir(FOLDER):
    raise RuntimeError(f'❌ AIgenerated folder not found: {FOLDER}')
if not os.path.isdir(ASSETS_ROOT):
    raise RuntimeError('❌ assets/ folder not found. Cleanup cancelled for safety.')

# Keep the stage2_outputs folder, but force-clear everything inside it.
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print('================================')
print('STAGE 2 COMPLETE CLEANUP')
print('================================')
print('Preserving: assets/')
print('Preserving folder: stage2_outputs/')
print()

# ------------------------------------------------------------
# 1. DIRECTLY EMPTY stage2_outputs/ AND KEEP THE FOLDER
# ------------------------------------------------------------
output_deleted=0
for name in list(os.listdir(OUTPUT_ROOT)):
    path=os.path.join(OUTPUT_ROOT, name)
    try:
        if os.path.isdir(path) and not os.path.islink(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
        output_deleted += 1
        print('🗑️ stage2_outputs:', name)
    except Exception as e:
        print(f'⚠️ Could not delete stage2_outputs/{name}: {e}')

# Verify that stage2_outputs is actually empty.
remaining_outputs=os.listdir(OUTPUT_ROOT)
if remaining_outputs:
    raise RuntimeError(
        '❌ stage2_outputs is NOT empty. Remaining items: '
        + ', '.join(remaining_outputs)
    )

# ------------------------------------------------------------
# 2. DELETE EVERYTHING ELSE IN AIgenerated/
#    EXCEPT assets/ AND stage2_outputs/
# ------------------------------------------------------------
kept={
    os.path.abspath(ASSETS_ROOT),
    os.path.abspath(OUTPUT_ROOT),
}

deleted=0
for name in list(os.listdir(FOLDER)):
    path=os.path.abspath(os.path.join(FOLDER, name))

    if path in kept:
        continue

    try:
        if os.path.isdir(path) and not os.path.islink(path):
            shutil.rmtree(path)
        else:
            os.remove(path)
        deleted += 1
        print('🗑️ workspace:', name)
    except Exception as e:
        print(f'⚠️ Could not delete {name}: {e}')

print()
print('================================')
print('CLEANUP COMPLETE')
print('================================')
print('Deleted workspace items:', deleted)
print('Cleared stage2_outputs items:', output_deleted)
print('✅ Kept: assets/')
print('✅ Kept: empty stage2_outputs/')
print('🧹 AIgenerated workspace is clean.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 2 COMPLETE CLEANUP
Preserving: assets/
Preserving folder: stage2_outputs/

🗑️ stage2_outputs: stage2_base.mp4
🗑️ workspace: 1SM_Cozy Bedroom Daydream in Pastels.jpg
🗑️ workspace: 9SM_Stuck in a Cozy Nighttime Thought.jpg
🗑️ workspace: 8SM_Cozy Bedroom Pout and Vanity.jpg
🗑️ workspace: 7SM_Rainy Night Bedroom Retreat.jpg
🗑️ workspace: 6SM_Cozy Nighttime Bedroom Reflection.jpg
🗑️ workspace: 5SM_IMG_20260902_122351.jpg
🗑️ workspace: 4SM_Dreamy Bedroom Memories.jpg
🗑️ workspace: 3SM_Peaceful Bedroom Meditation Moment.jpg
🗑️ workspace: 2SM_Reaching Across a Cozy Bedroom.jpg
🗑️ workspace: song.mp3
🗑️ workspace: _stage2_work

CLEANUP COMPLETE
Deleted workspace items: 11
Cleared stage2_outputs items: 1
✅ Kept: assets/
✅ Kept: empty stage2_outputs/
🧹 AIgenerated workspace is clean.
